**Análisis de Robustez entre Múltiples Semillas**

Importación principales 

In [1]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
import xgboost as xgb

Imports de validación y métricas

In [2]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.metrics import recall_score, make_scorer

Configuración y lista de semillas

In [3]:
CARPETA_RESULTADOS = "results"
SEMILLAS = [42, 7, 123, 2024, 99]   # 5 semillas distintas -> 5 ejecuciones completas

Creación del directorio de resultados

In [4]:
os.makedirs(CARPETA_RESULTADOS, exist_ok=True)

Carga del dataset y configuración inicial

In [5]:
X = pd.read_csv(f"{CARPETA_RESULTADOS}/X_full.csv")
y = pd.read_csv(f"{CARPETA_RESULTADOS}/y_full.csv").squeeze()
print(f"Dataset cargado: X={X.shape}, y={y.shape}")
print(f"Semillas a evaluar: {SEMILLAS}\n")
 
peso_clase_positiva = (y == 0).sum() / (y == 1).sum()
especificidad_scorer = make_scorer(recall_score, pos_label=0)
 
resultados_por_semilla = []  # una fila por (semilla, modelo)

Dataset cargado: X=(85, 17), y=(85,)
Semillas a evaluar: [42, 7, 123, 2024, 99]



# 1. Ejecución del bucle de semillas y validación cruzada

 Repetir TODA la CV robusta una vez por semilla

In [6]:
from tabpfn import TabPFNClassifier

for semilla in SEMILLAS:
    print(f"SEMILLA = {semilla}")

    cv_robusta = RepeatedStratifiedKFold(n_splits=10, n_repeats=5, random_state=semilla)

    pipelines = {
        'XGBoost': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('classifier', xgb.XGBClassifier(
                n_estimators=100, max_depth=3, learning_rate=0.1,
                scale_pos_weight=peso_clase_positiva,
                random_state=semilla, eval_metric='logloss'
            ))
        ]),
        'Regresión Logística': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('classifier', LogisticRegression(
                penalty='elasticnet', l1_ratio=0.5, solver='saga',
                class_weight='balanced', max_iter=5000, random_state=semilla
            ))
        ]),
        'Random Forest': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('classifier', RandomForestClassifier(
                n_estimators=200, max_depth=4,
                class_weight='balanced', random_state=semilla
            ))
        ]),
        'TabPFN': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('classifier', TabPFNClassifier(random_state=semilla))
        ])
    }

    scoring_dict = {
        'auc': 'roc_auc',
        'sensibilidad': 'recall',
        'especificidad': especificidad_scorer,
    }

    for nombre, pipe in pipelines.items():
        print(f"  Evaluando {nombre}...")
        cv_resultados = cross_validate(
            pipe, X, y, cv=cv_robusta, scoring=scoring_dict, n_jobs=1
        )
        scores_auc = cv_resultados['test_auc']
        scores_sens = cv_resultados['test_sensibilidad']
        scores_esp = cv_resultados['test_especificidad']

        print(f"  [{nombre}] AUC = {scores_auc.mean():.3f} (+/- {scores_auc.std():.3f})  "
              f"| Sens = {scores_sens.mean():.3f} | Esp = {scores_esp.mean():.3f}")

        resultados_por_semilla.append({
            'Semilla': semilla,
            'Modelo': nombre,
            'AUC_media': round(scores_auc.mean(), 4),
            'AUC_std_intra': round(scores_auc.std(), 4),
            'Sensibilidad_media': round(scores_sens.mean(), 4),
            'Especificidad_media': round(scores_esp.mean(), 4),
        })

    print()

SEMILLA = 42
  Evaluando XGBoost...
  [XGBoost] AUC = 0.820 (+/- 0.143)  | Sens = 0.633 | Esp = 0.793
  Evaluando Regresión Logística...
  [Regresión Logística] AUC = 0.867 (+/- 0.136)  | Sens = 0.820 | Esp = 0.790
  Evaluando Random Forest...
  [Random Forest] AUC = 0.851 (+/- 0.143)  | Sens = 0.607 | Esp = 0.839
  Evaluando TabPFN...
  [TabPFN] AUC = 0.873 (+/- 0.136)  | Sens = 0.693 | Esp = 0.825

SEMILLA = 7
  Evaluando XGBoost...
  [XGBoost] AUC = 0.821 (+/- 0.119)  | Sens = 0.693 | Esp = 0.783
  Evaluando Regresión Logística...
  [Regresión Logística] AUC = 0.864 (+/- 0.113)  | Sens = 0.780 | Esp = 0.789
  Evaluando Random Forest...
  [Random Forest] AUC = 0.838 (+/- 0.131)  | Sens = 0.653 | Esp = 0.826
  Evaluando TabPFN...
  [TabPFN] AUC = 0.859 (+/- 0.126)  | Sens = 0.693 | Esp = 0.808

SEMILLA = 123
  Evaluando XGBoost...
  [XGBoost] AUC = 0.803 (+/- 0.160)  | Sens = 0.713 | Esp = 0.788
  Evaluando Regresión Logística...
  [Regresión Logística] AUC = 0.864 (+/- 0.146)  | Sens

*Guardado del detalle por semilla*

  Guardar resultado detallado (una fila por semilla x modelo)

In [7]:
df_detalle = pd.DataFrame(resultados_por_semilla)
df_detalle.to_csv(f"{CARPETA_RESULTADOS}/robustez_semillas_detalle.csv", index=False)
print(f"[OK] Detalle guardado en '{CARPETA_RESULTADOS}/robustez_semillas_detalle.csv'")

[OK] Detalle guardado en 'results/robustez_semillas_detalle.csv'


# 2. Análisis y agregación entre semillas (Estabilidad del modelo)

Resumen de robustez entre semillas (AUC, sensibilidad y especificidad)

In [8]:
# 1. Agrupar y calcular media y desviación estándar
resumen_robustez = (
    df_detalle.groupby('Modelo')[['AUC_media', 'Sensibilidad_media', 'Especificidad_media']]
    .agg(['mean', 'std'])
    .round(4)
)

# 2. Crear un nuevo DataFrame con el símbolo de más-menos dentro del paréntesis
df_formateado = pd.DataFrame()
df_formateado['Modelo'] = resumen_robustez.index

for metrica_base, nombre_bonito in [
    ('AUC_media', 'AUC (Media ± Desv)'),
    ('Sensibilidad_media', 'Sensibilidad (Media ± Desv)'),
    ('Especificidad_media', 'Especificidad (Media ± Desv)')
]:
    media = resumen_robustez[(metrica_base, 'mean')]
    std = resumen_robustez[(metrica_base, 'std')]
    # Aquí añadimos el signo '±' antes de la desviación estándar
    df_formateado[nombre_bonito] = [f"{m:.4f} (±{s:.4f})" for m, s in zip(media, std)]

df_formateado['n_semillas'] = df_detalle.groupby('Modelo').size().values

# Ordenar por la media del AUC de forma descendente
df_formateado = df_formateado.sort_values(by='AUC (Media ± Desv)', ascending=False)

# Guardar y mostrar
df_formateado.to_csv(f"{CARPETA_RESULTADOS}/robustez_semillas_resumen_formateado.csv", index=False)

print("RESUMEN DE ROBUSTEZ ENTRE SEMILLAS")
print(df_formateado.to_string(index=False))

RESUMEN DE ROBUSTEZ ENTRE SEMILLAS
             Modelo AUC (Media ± Desv) Sensibilidad (Media ± Desv) Especificidad (Media ± Desv)  n_semillas
Regresión Logística   0.8644 (±0.0054)            0.8120 (±0.0218)             0.7874 (±0.0062)           5
             TabPFN   0.8618 (±0.0065)            0.7160 (±0.0209)             0.8177 (±0.0092)           5
      Random Forest   0.8476 (±0.0075)            0.6120 (±0.0237)             0.8283 (±0.0060)           5
            XGBoost   0.8168 (±0.0091)            0.6866 (±0.0309)             0.7931 (±0.0090)           5
